# 6-amaliyot: Ma'lumotlarni birlashtirish va qayta shakllantirish

**Data Analysis kursi** · boshlang'ich daraja · taxminiy vaqt: 80 daqiqa

Bu amaliyotda 6-ma'ruzadagi amallarni qo'lda bajarib ko'ramiz:

| Bo'lim | Mavzu | Asosiy funksiya |
|---|---|---|
| 1 | Jadvallarni ustma-ust va yonma-yon qo'shish | `pd.concat` |
| 2 | Jadvallarni kalit ustun bo'yicha birlashtirish | `pd.merge` |
| 3 | Indeks bo'yicha birlashtirish | `df.join` |
| 4 | Uzun jadvaldan xulosaviy jadval | `pivot`, `pivot_table` |
| 5 | Keng jadvalni uzun qilish | `melt` |
| 6 | Ustun va indeks orasida aylantirish | `stack`, `unstack` |
| 7 | Mini-loyiha: do'kon hisoboti | hammasi birga |

### Qanday ishlaymiz
1. Har bir bo'limda avval **misol** keltiriladi — uni ishga tushiring va natijani o'qing.
2. Keyin **🔹 Mashq** keladi — `# Kodingizni shu yerga yozing` degan joyni to'ldiring.
3. Mashqdan keyingi ** Tekshirish** katakchasini ishga tushiring. Agar `To'g'ri!` chiqsa — keyingi mashqqa o'ting.
   Xato bo'lsa, xabarni o'qing va kodni tuzating.

> Katakchani ishga tushirish: **Shift + Enter**

In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.width', 120)

---
## 1. `pd.concat` — jadvallarni qo'shish

`concat` bir xil tuzilishdagi jadvallarni **ustma-ust** (qator bo'yicha) yoki **yonma-yon** (ustun bo'yicha) qo'shadi.

```python
pd.concat([df1, df2])                     # ustma-ust (axis=0, standart)
pd.concat([df1, df2], ignore_index=True)  # indeksni 0, 1, 2, ... qilib qayta raqamlash
pd.concat([df1, df2], axis=1)             # yonma-yon
```

**Misol:** do'konning yanvar va fevral oylari savdosi alohida jadvalda kelgan.

In [4]:
yanvar = pd.DataFrame({
    'mahsulot': ['Non', 'Sut', 'Tuxum'],
    'soni':     [120, 80, 200]
})
fevral = pd.DataFrame({
    'mahsulot': ['Non', 'Sut', 'Tuxum'],
    'soni':     [130, 75, 210]
})

# yanvar va fevral jadvallarini alohid ko'ring

,mahsulot,soni
0,Non,120
1,Sut,80
2,Tuxum,200
0,Non,130
1,Sut,75
2,Tuxum,210


In [5]:
ikki_oy = pd.concat([yanvar, fevral])
ikki_oy

,mahsulot,soni
0,Non,120
1,Sut,80
2,Tuxum,200
0,Non,130
1,Sut,75
2,Tuxum,210


**E'tibor bering:** indeks `0, 1, 2, 0, 1, 2` — takrorlanib qoldi. Buni tuzatish uchun `ignore_index=True` ishlatamiz.
Qaysi qator qaysi oydan kelganini yo'qotmaslik uchun esa avval `oy` ustunini qo'shib olamiz:

In [6]:
yanvar['oy'] = 'Yanvar'
fevral['oy'] = 'Fevral'

ikki_oy = pd.concat([yanvar, fevral], ignore_index=True)
ikki_oy

,mahsulot,soni,oy
0,Non,120,Yanvar
1,Sut,80,Yanvar
2,Tuxum,200,Yanvar
3,Non,130,Fevral
4,Sut,75,Fevral
5,Tuxum,210,Fevral


### 🔹 Mashq 1.1
Quyida mart oyi savdosi berilgan. `mart` jadvaliga `oy` ustunini qo'shing (qiymati `'Mart'`),
so'ng `yanvar`, `fevral` va `mart` jadvallarini **bitta** `uch_oy` jadvaliga birlashtiring. Indeks `0` dan `8` gacha bo'lsin.

In [7]:
mart = pd.DataFrame({
    'mahsulot': ['Non', 'Sut', 'Tuxum'],
    'soni':     [125, 90, 190]
})

In [8]:
# Kodingizni shu yerga yozing
mart['oy'] = ...
uch_oy = ...
uch_oy

Ellipsis

In [9]:
#  Tekshirish
assert uch_oy.shape == (9, 3), f"Jadval o'lchami (9, 3) bo'lishi kerak, sizda: {uch_oy.shape}"
assert list(uch_oy.index) == list(range(9)), "Indeks 0..8 bo'lishi kerak — ignore_index=True ni unutmang"
assert uch_oy['soni'].sum() == 1220,  "Yig'indi 1220 bo'lishi kerak"
print("To'g'ri! ")

AttributeError: 'ellipsis' object has no attribute 'shape'

### 🔹 Mashq 1.2
Har bir mahsulot uchun **narx** va **ombordagi qoldiq** alohida jadvalda berilgan.
Ikkalasini **yonma-yon** (`axis=1`) birlashtirib, `mahsulot_info` jadvalini hosil qiling.

In [ ]:
narxlar = pd.DataFrame({'narx': [4000, 12000, 1500]},
                       index=['Non', 'Sut', 'Tuxum'])
qoldiq  = pd.DataFrame({'qoldiq': [50, 20, 300]},
                       index=['Non', 'Sut', 'Tuxum'])

In [ ]:
# Kodingizni shu yerga yozing
mahsulot_info = ...
mahsulot_info

In [ ]:
#  Tekshirish
assert list(mahsulot_info.columns) == ['narx', 'qoldiq'], "Ustunlar: ['narx', 'qoldiq'] bo'lishi kerak"
assert mahsulot_info.loc['Sut', 'qoldiq'] == 20
print("To'g'ri! ")

---
## 2. `pd.merge` — kalit ustun bo'yicha birlashtirish

`merge` ikki jadvalni **umumiy ustun** (kalit) bo'yicha birlashtiradi — xuddi SQL'dagi `JOIN` kabi.

| `how=` | Natijada qaysi qatorlar qoladi |
|---|---|
| `'inner'` (standart) | faqat **ikkala** jadvalda bor kalitlar |
| `'left'` | **chap** jadvalning hamma qatorlari |
| `'right'` | **o'ng** jadvalning hamma qatorlari |
| `'outer'` | **ikkala** jadvalning hamma qatorlari |

Topilmagan joylarga `NaN` yoziladi.

**Misol:** talabalar va ularning imtihon natijalari.

In [ ]:
talabalar = pd.DataFrame({
    'id':   [1, 2, 3, 4],
    'ism':  ['Aziz', 'Dilnoza', 'Jasur', 'Malika'],
    'guruh': ['A', 'B', 'A', 'B']
})
baholar = pd.DataFrame({
    'id':   [1, 2, 3, 5],
    'ball': [85, 92, 78, 66]
})
print("Talabalar:"); display(talabalar)
print("Baholar:");   display(baholar)

In [ ]:
# inner: faqat ikkala jadvalda bor id'lar (1, 2, 3)
pd.merge(talabalar, baholar, on='id')

In [ ]:
# left: barcha talabalar qoladi; Malika (id=4) bahosi yo'q -> NaN
pd.merge(talabalar, baholar, on='id', how='left')

In [ ]:
# outer + indicator: har bir qator qayerdan kelganini ko'rsatadi
pd.merge(talabalar, baholar, on='id', how='outer', indicator=True)

### 🔹 Mashq 2.1 — bashorat qiling, keyin tekshiring
Kodni ishga tushirishdan **oldin** har bir join turi uchun natijada nechta qator bo'lishini o'ylab, javobni yozing.

In [ ]:
# Javoblaringizni raqam bilan yozing
inner_soni = ...
left_soni  = ...
right_soni = ...
outer_soni = ...

In [ ]:
#  Tekshirish
for nom, javob in [('inner', inner_soni), ('left', left_soni), ('right', right_soni), ('outer', outer_soni)]:
    haqiqiy = len(pd.merge(talabalar, baholar, on='id', how=nom))
    belgi = "✅" if javob == haqiqiy else "❌"
    print(f"{belgi} {nom:6s}: sizning javob = {javob}, haqiqiy = {haqiqiy}")

### 🔹 Mashq 2.2
Har bir **talabaning** ismi va bali chiqadigan `natija` jadvalini yarating.
Bahosi yo'q talabalar ham ro'yxatda qolsin. Qaysi birlashtirish usuli kerak bo'ladi?

In [ ]:
# Kodingizni shu yerga yozing
natija = ...
natija

In [ ]:
#  Tekshirish
assert len(natija) == 4, "Barcha 4 ta talaba qolishi kerak"
assert natija['ball'].isna().sum() == 1, "Bitta talabaning bahosi NaN bo'lishi kerak (Malika)"
print("To'g'ri! ")

### Ustun nomlari har xil bo'lsa: `left_on` va `right_on`
Ba'zan kalit ustunlar boshqacha nomlanadi: bir jadvalda `id`, boshqasida `talaba_id`.

In [ ]:
davomat = pd.DataFrame({
    'talaba_id': [1, 2, 3, 4],
    'qoldirgan_dars': [0, 3, 1, 6]
})
pd.merge(talabalar, davomat, left_on='id', right_on='talaba_id')

### 🔹 Mashq 2.3
`talabalar` va `davomat` jadvallarini birlashtiring, so'ng **3 tadan ortiq** dars qoldirgan talabalarning faqat ismini chiqaring.

In [ ]:
# Kodingizni shu yerga yozing
birga = ...
kop_qoldirgan = ...
kop_qoldirgan

In [ ]:
#  Tekshirish
assert list(kop_qoldirgan) == ['Malika'], f"Javob ['Malika'] bo'lishi kerak, sizda: {list(kop_qoldirgan)}"
print("To'g'ri! ")

---
## 3. `df.join` — indeks bo'yicha birlashtirish

Agar kalit **indeksda** tursa, `join` qisqaroq yoziladi. Standart holatda u **left join** qiladi.

In [ ]:
shaharlar = pd.DataFrame({'shahar': ['Toshkent', 'Samarqand', 'Buxoro', 'Toshkent']},
                         index=[1, 2, 3, 4])
talabalar_i = talabalar.set_index('id')   # id ni indeksga o'tkazamiz

talabalar_i.join(shaharlar)

### 🔹 Mashq 3.1
Quyidagi `telefon` jadvalini `talabalar_i` ga `join` bilan qo'shing. Natijani `toliq` deb nomlang.

In [ ]:
telefon = pd.DataFrame({'telefon': ['90-111-22-33', '93-444-55-66']},
                       index=[1, 3])

In [ ]:
# Kodingizni shu yerga yozing
toliq = ...
toliq

In [ ]:
#  Tekshirish
assert len(toliq) == 4, "join standart holatda left join — 4 ta qator qolishi kerak"
assert toliq['telefon'].isna().sum() == 2, "2 ta talabaning telefoni NaN bo'lishi kerak"
print("To'g'ri! ")

---
## 4. `pivot` va `pivot_table` — xulosaviy jadval

**Uzun (long)** jadvaldan **keng (wide)** jadval yasaymiz: bitta ustun qiymatlari yangi **ustunlarga** aylanadi.

```python
df.pivot(index='qator_uchun', columns='ustun_uchun', values='qiymat')
df.pivot_table(index=..., columns=..., values=..., aggfunc='sum')   # takrorlansa — yig'adi
```

**Qoida:** agar `(index, columns)` juftligi takrorlansa, `pivot` **xato** beradi — u holda `pivot_table` kerak.

In [ ]:
savdo = pd.DataFrame({
    'filial': ['Chilonzor', 'Chilonzor', 'Yunusobod', 'Yunusobod', 'Chilonzor', 'Yunusobod'],
    'oy':     ['Yanvar',    'Fevral',    'Yanvar',    'Fevral',    'Mart',      'Mart'],
    'tushum': [120,         135,         98,          110,         140,         105]
})
savdo

In [ ]:
# Har bir (filial, oy) juftligi bir marta uchraydi -> pivot ishlaydi
savdo.pivot(index='filial', columns='oy', values='tushum')

Endi bir kunda bir nechta chek bo'lgan jadvalni olaylik. Bu yerda `pivot` ishlamaydi — `pivot_table` yig'indini hisoblaydi:

In [ ]:
cheklar = pd.DataFrame({
    'kassir':  ['Ali', 'Ali', 'Vali', 'Ali', 'Vali', 'Vali'],
    'kun':     ['Du',  'Du',  'Du',   'Se',  'Se',   'Se'],
    'summa':   [50,    30,    40,     20,    60,     10]
})
cheklar.pivot_table(index='kassir', columns='kun', values='summa', aggfunc='sum')

### 🔹 Mashq 4.1
`cheklar` jadvalidan har bir kassir har kuni **nechta chek** urganini ko'rsatuvchi jadval tuzing
(`aggfunc='count'`). Natijani `chek_soni` deb nomlang.

In [ ]:
# Kodingizni shu yerga yozing
chek_soni = ...
chek_soni

In [ ]:
#  Tekshirish
assert chek_soni.loc['Ali', 'Du'] == 2 and chek_soni.loc['Vali', 'Se'] == 2
print("To'g'ri! ")

### 🔹 Mashq 4.2
`savdo` jadvalidan har bir **oy** uchun filiallar tushumini ko'rsating — bu safar **oylar qatorda**, **filiallar ustunda** bo'lsin.
Qo'shimcha: `margins=True` qo'shib, jami qator va ustunni ham chiqaring.

In [ ]:
# Kodingizni shu yerga yozing
oylik = ...
oylik

In [ ]:
#  Tekshirish
assert 'Chilonzor' in oylik.columns, "Filiallar ustunda bo'lishi kerak"
assert oylik.loc['All', 'All'] == 708, "Jami tushum 708 bo'lishi kerak — margins=True ni qo'shdingizmi?"
print("To'g'ri! ")

---
## 5. `melt` — keng jadvalni uzun qilish

`melt` — `pivot`ning **teskarisi**. Ustun nomlari qiymatga aylanadi.

```python
df.melt(id_vars='saqlanadigan_ustun', var_name='yangi_ustun_nomi', value_name='qiymat_nomi')
```

In [ ]:
jurnal = pd.DataFrame({
    'ism':        ['Aziz', 'Dilnoza', 'Jasur'],
    'matematika': [85, 92, 70],
    'fizika':     [78, 88, 65],
    'ingliz':     [90, 95, 80]
})
jurnal

In [ ]:
uzun = jurnal.melt(id_vars='ism', var_name='fan', value_name='ball')
uzun

Endi uzun jadvalda `groupby` bilan har bir fan bo'yicha o'rtacha ballni oson topamiz:

In [ ]:
uzun.groupby('fan')['ball'].mean()

### 🔹 Mashq 5.1
Quyidagi `harorat` jadvalini uzun shaklga keltiring: ustunlar `shahar`, `kun`, `harorat` bo'lsin.
So'ng har bir shaharning **eng yuqori** haroratini toping.

In [ ]:
harorat = pd.DataFrame({
    'shahar': ['Toshkent', 'Samarqand', 'Nukus'],
    'Du': [18, 16, 12],
    'Se': [21, 17, 10],
    'Ch': [19, 20, 14]
})

In [ ]:
# Kodingizni shu yerga yozing
harorat_uzun = ...
eng_issiq = ...
eng_issiq

In [ ]:
# ✅ Tekshirish
assert harorat_uzun.shape == (9, 3), f"Uzun jadval (9, 3) bo'lishi kerak, sizda: {harorat_uzun.shape}"
assert list(harorat_uzun.columns) == ['shahar', 'kun', 'harorat']
assert eng_issiq['Samarqand'] == 20 and eng_issiq['Nukus'] == 14
print("To'g'ri! ")

---
## 6. `stack` va `unstack`

- `stack()` — ustunlarni **indeksning ichki darajasiga** tushiradi (keng → uzun, natija ko'pincha `Series`).
- `unstack()` — indeksning ichki darajasini **ustunlarga** ko'taradi (uzun → keng).

In [ ]:
keng = savdo.pivot(index='filial', columns='oy', values='tushum')
keng

In [ ]:
s = keng.stack()      # ikki darajali indeks: (filial, oy)
s

In [ ]:
s.unstack()           # qaytadan keng jadval

In [ ]:
s.unstack(level=0)    # endi filiallar ustunga chiqadi

### 🔹 Mashq 6.1
`s` dan foydalanib, **Yunusobod** filialining **Fevral** oyidagi tushumini toping.
Yordam: ikki darajali indeksda
`s.loc[('birinchi', 'ikkinchi')]` deb murojaat qilinadi.

In [ ]:
# Kodingizni shu yerga yozing
yunusobod_fevral = ...
yunusobod_fevral

In [ ]:
#  Tekshirish
assert yunusobod_fevral == 110
print("To'g'ri! ")

---
## 7. Mini-loyiha: do'kon hisoboti

Do'kon tarmog'ida ikki filial bor. Har bir filial savdosi alohida jadvalda, mahsulotlar narxi esa boshqa jadvalda.

**Vazifa — 4 qadam:**
1. `chilonzor` va `yunusobod` jadvallariga `filial` ustunini qo'shib, `concat` bilan bitta `savdolar` jadvaliga birlashtiring.
2. `savdolar` ni `mahsulotlar` jadvali bilan `merge` qiling (kalit: `mahsulot`).
3. Yangi `tushum = soni * narx` ustunini hisoblang.
4. `pivot_table` bilan **mahsulot × filial** bo'yicha jami tushum jadvalini (`hisobot`) tuzing.

In [ ]:
chilonzor = pd.DataFrame({
    'mahsulot': ['Non', 'Sut', 'Non', 'Tuxum'],
    'soni':     [10, 5, 8, 30]
})
yunusobod = pd.DataFrame({
    'mahsulot': ['Sut', 'Non', 'Tuxum', 'Sut'],
    'soni':     [7, 12, 20, 3]
})
mahsulotlar = pd.DataFrame({
    'mahsulot': ['Non', 'Sut', 'Tuxum'],
    'narx':     [4000, 12000, 1500]
})

In [ ]:
# 1-qadam
chilonzor['filial'] = ...
yunusobod['filial'] = ...
savdolar = ...

# 2-qadam
toliq_savdo = ...

# 3-qadam
toliq_savdo['tushum'] = ...

# 4-qadam
hisobot = ...
hisobot

In [ ]:
#  Tekshirish
assert len(savdolar) == 8, "1-qadam: savdolar jadvalida 8 ta qator bo'lishi kerak"
assert 'narx' in toliq_savdo.columns, "2-qadam: narx ustuni qo'shilmagan"
assert toliq_savdo['tushum'].sum() == 375000, "3-qadam: jami tushum 375 000 bo'lishi kerak"
assert hisobot.loc['Non', 'Chilonzor'] == 72000, "4-qadam: Chilonzorda non tushumi 72 000 bo'lishi kerak"
assert hisobot.loc['Sut', 'Yunusobod'] == 120000
print("Tabriklaymiz! Mini-loyiha to'liq bajarildi 🎉")

### 🔹 Mustahkamlovchi savollar (o'z so'zingiz bilan javob yozing)
1. Qaysi filial ko'proq tushum qildi? Buni `hisobot.sum()` bilan tekshiring.
2. 2-qadamda `how='left'` o'rniga `how='inner'` yozsak, natija o'zgarardimi? Nega?
3. `hisobot` jadvalini `melt` yoki `stack` bilan qayta uzun shaklga keltiring.

In [ ]:
# Kodingizni shu yerga yozing

---
##  **Xulosa  **

| Vazifa | Funksiya |
|---|---|
| Bir xil jadvallarni ustma-ust qo'shish | `pd.concat([...], ignore_index=True)` |
| Kalit ustun bo'yicha birlashtirish | `pd.merge(a, b, on='kalit', how='left')` |
| Indeks bo'yicha birlashtirish | `a.join(b)` |
| Uzun → keng (takrorsiz) | `df.pivot(index, columns, values)` |
| Uzun → keng (yig'ib) | `df.pivot_table(..., aggfunc='sum')` |
| Keng → uzun | `df.melt(id_vars=...)` |
| Ustun ↔ indeks | `stack()` / `unstack()` |

**Eng ko'p uchraydigan xatolar:**
- `concat`dan keyin `ignore_index=True` ni unutish → takroriy indeks
- `merge`da `how=` ni yozmaslik → standart `inner` jimgina qatorlarni o'chiradi
- takroriy juftlikda `pivot` ishlatish → `ValueError: Index contains duplicate entries`